In [3]:

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import json, time, zipfile

START = time.time()
SOURCE = Path("/kaggle/input/datasets/pranavsunilwagh/nd-iris/NDCLD15")
PROJECT = Path("/kaggle/working/nd_iris_project")
OUT = PROJECT / "notebook02_preprocessing"
IMAGES_OUT = OUT / "images_224"
IMAGES_OUT.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists(), f"Dataset not found: {SOURCE}"

# Reuse Notebook 01 inventory if available; otherwise discover TIFFs.
inventory = PROJECT / "notebook01_dataset_analysis/ND_Iris_file_metadata.csv"
if inventory.exists():
    source_paths = [
        SOURCE / rel for rel in
        pd.read_csv(inventory)["relative_path"].dropna().tolist()
    ]
    source_paths = [p for p in source_paths if p.is_file()]
else:
    source_paths = sorted(SOURCE.rglob("*.tiff"))

source_paths = sorted(set(source_paths))
print(f"Source images found: {len(source_paths):,}")

metadata, errors = [], []
for src in tqdm(source_paths, desc="Preprocessing"):
    rel = src.relative_to(SOURCE)
    # Unique output name preserves nested-folder distinction.
    out_name = "__".join(rel.parts[:-1] + (src.stem,)) + ".png"
    dst = IMAGES_OUT / out_name

    try:
        with Image.open(src) as im:
            original_width, original_height = im.size
            original_mode = im.mode
            processed = im.convert("L").resize(
                (224, 224), Image.Resampling.LANCZOS
            )
            processed.save(dst, format="PNG")

        metadata.append({
            "filename": src.name,
            "relative_path": str(rel),
            "processed_path": str(dst),
            "processed_filename": out_name,
            "original_width": original_width,
            "original_height": original_height,
            "original_mode": original_mode,
            "processed_width": 224,
            "processed_height": 224,
            "processed_mode": "L",
            "valid": True,
        })
    except Exception as e:
        errors.append({"relative_path": str(rel), "error": str(e)[:300]})

meta = pd.DataFrame(metadata)
meta.to_csv(OUT / "ND_Iris_preprocessed_metadata.csv", index=False)
pd.DataFrame(errors, columns=["relative_path", "error"]).to_csv(
    OUT / "ND_Iris_preprocessing_errors.csv", index=False
)

# Verify saved outputs.
verified, verify_errors = 0, []
for row in tqdm(metadata, desc="Verifying outputs"):
    path = Path(row["processed_path"])
    try:
        with Image.open(path) as im:
            assert im.size == (224, 224)
            assert im.mode == "L"
            im.verify()
        verified += 1
    except Exception as e:
        verify_errors.append({"file": str(path), "error": str(e)[:300]})

pd.DataFrame(verify_errors, columns=["file", "error"]).to_csv(
    OUT / "ND_Iris_output_verification_errors.csv", index=False
)

summary = {
    "source_images_found": len(source_paths),
    "successfully_preprocessed": len(metadata),
    "preprocessing_failures": len(errors),
    "verified_output_images": verified,
    "output_verification_failures": len(verify_errors),
    "output_size": [224, 224],
    "output_mode": "L",
    "identity_labels_verified": False,
    "identity_labels_created": False,
}
with open(OUT / "ND_Iris_preprocessing_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

conclusion = f"""# Notebook 02 — ND Iris Image Preprocessing: Conclusion

- Source images discovered: **{len(source_paths):,}**
- Successfully preprocessed: **{len(metadata):,}**
- Processing failures: **{len(errors):,}**
- Output images verified: **{verified:,}**
- Verification failures: **{len(verify_errors):,}**
- Output format: grayscale PNG, 224 × 224 pixels.

All successfully processed images were converted to grayscale and resized using
Lanczos resampling. Source-relative paths and original dimensions were preserved
in the metadata. This notebook does not infer or create identity labels; those
must be validated separately before recognition experiments.
"""
(OUT / "Notebook02_Conclusion.md").write_text(conclusion)

zip_path = PROJECT / "ND_Iris_Notebook02_Results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    # Include metadata/reports, not thousands of processed images.
    # Images remain in Kaggle working storage for subsequent notebooks.
    for p in OUT.rglob("*"):
        if p.is_file() and IMAGES_OUT not in p.parents:
            z.write(p, arcname=f"notebook02_preprocessing/{p.relative_to(OUT)}")

with zipfile.ZipFile(zip_path) as z:
    bad = z.testzip()

print("\n" + "=" * 65)
print("NOTEBOOK 02 RESULTS")
print("=" * 65)
print(json.dumps(summary, indent=2))
print(f"Processed image directory: {IMAGES_OUT}")
print(f"Metadata: {OUT / 'ND_Iris_preprocessed_metadata.csv'}")
print(f"Conclusion: {OUT / 'Notebook02_Conclusion.md'}")
print(f"ZIP: {zip_path}")
print(f"ZIP integrity: {'PASSED' if bad is None else 'FAILED: ' + str(bad)}")
print(f"Elapsed: {(time.time()-START)/60:.2f} minutes")


Source images found: 6,233


Preprocessing:   0%|          | 0/6233 [00:00<?, ?it/s]

Verifying outputs:   0%|          | 0/6233 [00:00<?, ?it/s]


NOTEBOOK 02 RESULTS
{
  "source_images_found": 6233,
  "successfully_preprocessed": 6233,
  "preprocessing_failures": 0,
  "verified_output_images": 6233,
  "output_verification_failures": 0,
  "output_size": [
    224,
    224
  ],
  "output_mode": "L",
  "identity_labels_verified": false,
  "identity_labels_created": false
}
Processed image directory: /kaggle/working/nd_iris_project/notebook02_preprocessing/images_224
Metadata: /kaggle/working/nd_iris_project/notebook02_preprocessing/ND_Iris_preprocessed_metadata.csv
Conclusion: /kaggle/working/nd_iris_project/notebook02_preprocessing/Notebook02_Conclusion.md
ZIP: /kaggle/working/nd_iris_project/ND_Iris_Notebook02_Results.zip
ZIP integrity: PASSED
Elapsed: 1.57 minutes


## Conclusion — Notebook 02: Image Preprocessing

The ND Iris dataset containing **6,233 TIFF images** was successfully preprocessed. All images were converted to grayscale and resized to **224 × 224 pixels** using Lanczos resampling.

### Results

- **Total source images:** 6,233
- **Successfully preprocessed:** 6,233
- **Preprocessing failures:** 0
- **Output images verified:** 6,233
- **Output verification failures:** 0
- **Output format:** Grayscale PNG
- **Output dimensions:** 224 × 224 pixels
- **ZIP integrity:** Passed

### Key Findings

All source images were processed successfully, and every saved output image passed verification. The preprocessing metadata preserves the original filenames, source-relative paths, and original image dimensions.

Identity labels have not yet been independently verified or created. Therefore, identity recognition experiments should not begin until the dataset's identity structure has been established.

### Generated Outputs

- `ND_Iris_preprocessed_metadata.csv`
- `ND_Iris_preprocessing_errors.csv`
- `ND_Iris_output_verification_errors.csv`
- `ND_Iris_preprocessing_summary.json`
- `images_224/` — all 6,233 preprocessed images
- `ND_Iris_Notebook02_Results.zip` — archived notebook outputs

**Overall conclusion:** Notebook 02 successfully completed image preprocessing and output verification with zero reported failures. The resulting images are ready for the next stage of ocular image quality assessment, subject to subsequent identity-label validation.